# 11 - Financial Research Kit

This notebook is a read-only financial research starter for Atlas. It uses OpenBB and CCXT for market data exploration, keeps portfolio examples paper-only, and stores derived datasets in MinIO when available.

Safety boundary: this is not financial advice, it performs no live trading, and it intentionally blocks live exchange credentials and private/trading CCXT methods in the first Atlas trading slice.


In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd

from atlas_finance.research import (
    assert_no_live_exchange_credentials,
    assert_public_ccxt_method,
    make_public_exchange_config,
    paper_portfolio_summary,
)

assert_no_live_exchange_credentials(os.environ)
for method_name in ["load_markets", "fetch_ticker", "fetch_ohlcv", "fetch_order_book"]:
    assert_public_ccxt_method(method_name)

print("Financial research guardrails active: read-only data, paper portfolios, no live trading.")


## 1. Atlas service endpoints

The notebook can use these Atlas-provided environment variables when the related services are enabled:

- `AWS_ENDPOINT_URL_S3`, `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY` for MinIO dataset writes.
- `MLFLOW_TRACKING_URI` for optional paper-run experiment logging.
- `LITELLM_BASE_URL`, `OPENAI_API_BASE`, and `OPENAI_API_KEY` for optional LiteLLM summaries.

Do not add exchange API keys to this notebook or to `.env` for this slice. Future exchange-key workflows require a secrets-management service and explicit risk/audit controls.


In [ ]:
for key in [
    "AWS_ENDPOINT_URL_S3",
    "MLFLOW_TRACKING_URI",
    "LITELLM_BASE_URL",
    "OPENAI_API_BASE",
]:
    print(f"{key}={os.getenv(key, '<unset>')}")


## 2. Public market data with OpenBB

OpenBB is available in the JupyterHub image for read-only research. Provider availability depends on the installed OpenBB provider packages and whether a provider requires its own key. Keep examples public by default and treat network/API-key failures as non-blocking for local Atlas demos.


In [ ]:
try:
    from openbb import obb

    # Public provider availability can vary; this cell is deliberately optional.
    data = obb.equity.price.historical(symbol="AAPL", provider="yfinance").to_df()
    display(data.tail())
except Exception as exc:
    print(f"OpenBB public-data example skipped: {exc}")


## 3. Public crypto market data with CCXT

CCXT is included for unified public exchange data. This notebook only calls public methods such as `load_markets`, `fetch_ticker`, `fetch_ohlcv`, and `fetch_order_book`. Private account, transfer, withdrawal, and order-placement methods remain blocked.


In [ ]:
try:
    import ccxt

    exchange = ccxt.binance(make_public_exchange_config())
    markets = exchange.load_markets()
    ticker = exchange.fetch_ticker("BTC/USDT")
    print(f"Loaded {len(markets)} public markets")
    print({key: ticker.get(key) for key in ["symbol", "last", "datetime"]})
except Exception as exc:
    print(f"CCXT public-data example skipped: {exc}")


## 4. Paper portfolio analysis

The helper computes a paper portfolio summary from local positions and mark prices. It does not connect to accounts, place orders, or read exchange balances.


In [ ]:
positions = [
    {"symbol": "BTC/USDT", "quantity": 0.10, "cost_basis": 60000.0},
    {"symbol": "ETH/USDT", "quantity": 1.50, "cost_basis": 3000.0},
]
marks = {"BTC/USDT": 64000.0, "ETH/USDT": 2800.0}

summary = paper_portfolio_summary(positions=positions, marks=marks)
summary_df = pd.DataFrame(summary["positions"])
display(summary_df)
print({key: summary[key] for key in ["total_market_value", "total_cost_basis", "unrealized_pnl"]})


## 5. Persist derived datasets to MinIO

When MinIO is enabled, derived paper-portfolio datasets can be written to an Atlas bucket through the S3-compatible endpoint. This keeps research outputs available to Spark, Airflow, MLflow, and downstream notebooks without introducing a trading database in this first slice.


In [ ]:
dataset_path = Path('/tmp/atlas-paper-portfolio-summary.parquet')
summary_df.assign(snapshot_at=datetime.now(timezone.utc).isoformat()).to_parquet(dataset_path, index=False)
print(f"Wrote local parquet dataset: {dataset_path}")

try:
    import boto3

    endpoint = os.getenv("AWS_ENDPOINT_URL_S3")
    access_key = os.getenv("AWS_ACCESS_KEY_ID")
    secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
    bucket = os.getenv("ATLAS_FINANCE_BUCKET", os.getenv("MINIO_BUCKET_JUPYTER", "jupyter"))

    if not endpoint or not access_key or not secret_key:
        raise RuntimeError("MinIO env is not fully configured")

    s3 = boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
    )
    try:
        s3.head_bucket(Bucket=bucket)
    except Exception as exc:
        response = getattr(exc, "response", None)
        if not isinstance(response, dict):
            raise
        error = response.get("Error")
        metadata = response.get("ResponseMetadata")
        code = str(error.get("Code") or "") if isinstance(error, dict) else ""
        status = metadata.get("HTTPStatusCode") if isinstance(metadata, dict) else None
        if status != 404 and code not in {"404", "NoSuchBucket", "NotFound"}:
            raise
        s3.create_bucket(Bucket=bucket)

    key = f"paper-portfolios/{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.parquet"
    s3.upload_file(str(dataset_path), bucket, key)
    print(f"Uploaded s3://{bucket}/{key}")
except Exception as exc:
    print(f"MinIO upload skipped: {exc}")


## 6. Optional MLflow and LiteLLM handoff

MLflow can record paper-run parameters and metrics when enabled. LiteLLM can summarize the run for a notebook narrative, but summaries must stay descriptive and must not become trading recommendations.


In [ ]:
try:
    import mlflow

    tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
    if not tracking_uri:
        raise RuntimeError("MLFLOW_TRACKING_URI is not set")

    mlflow.set_tracking_uri(tracking_uri)
    mlflow.set_experiment("atlas-financial-research-kit")
    with mlflow.start_run(run_name="paper-portfolio-smoke"):
        mlflow.log_param("mode", "paper-only")
        mlflow.log_metric("total_market_value", float(summary["total_market_value"]))
        mlflow.log_metric("unrealized_pnl", float(summary["unrealized_pnl"]))
    print("Logged paper portfolio metrics to MLflow")
except Exception as exc:
    print(f"MLflow logging skipped: {exc}")


In [ ]:
try:
    from openai import OpenAI

    base_url = os.getenv("OPENAI_API_BASE") or os.getenv("LITELLM_BASE_URL")
    api_key = os.getenv("OPENAI_API_KEY", "sk-atlas-local")
    if not base_url:
        raise RuntimeError("LiteLLM env is not configured")

    client = OpenAI(base_url=base_url, api_key=api_key)
    response = client.chat.completions.create(
        model=os.getenv("ATLAS_SUMMARY_MODEL", "ollama/qwen3.8:latest"),
        messages=[
            {"role": "system", "content": "Summarize paper portfolio diagnostics. Do not provide financial advice."},
            {"role": "user", "content": str({key: summary[key] for key in ["total_market_value", "total_cost_basis", "unrealized_pnl"]})},
        ],
    )
    print(response.choices[0].message.content)
except Exception as exc:
    print(f"LiteLLM summary skipped: {exc}")


## 7. Next steps

Good follow-up tickets can add curated public datasets, TimescaleDB evaluation, secrets-management-backed exchange-key workflows, and sandbox-only execution engines. Keep each addition explicit about tracks, ports, credentials, topology edges, and whether it is disabled by default.
